# CiteFocus Lexical Search: Inner Workings

This notebook is intentionally **not** a black-box wrapper.

It shows, cell by cell:

1. how the lexical query words are built
2. how the SQLite search over `title_words` and `records` works
3. how candidates are scored
4. how the final lexical ranking is produced

We use one real citation from your local parsed JSON and query all three local DBs:

- arXiv
- DBLP
- OpenAlex

## 1. Imports and paths

In [ ]:
import json
import re
import sqlite3
from pathlib import Path
from pprint import pprint

ROOT = Path('/home/sascha/refcheck/CiteFocus')
PARSED_PATH = ROOT / 'outputs' / 'parsed_citations_gv_20260401_160815.json'
ARXIV_DB = ROOT / 'academic_metadata' / 'arxiv' / 'oai_pages' / 'arxiv_local_index.sqlite'
DBLP_DB = ROOT / 'academic_metadata' / 'dblp' / 'dblp_local_index.sqlite'
OPENALEX_DB = ROOT / 'academic_metadata' / 'openalex' / 'openalex_local_index.sqlite'

print(PARSED_PATH)
print(ARXIV_DB)
print(DBLP_DB)
print(OPENALEX_DB)

## 2. Pick one example citation

We use citation `9` because it is clean and easy to reason about.

In [ ]:
parsed = json.loads(PARSED_PATH.read_text())
citation = next(r for r in parsed if r['citation_id'] == 9)
pprint(citation)

## 3. Rebuild the query-word logic in the notebook

Below is a simplified copy of the lexical agent logic. The point is to make the transformation visible.

In [ ]:
STOP_WORDS = {'a', 'an', 'the', 'of', 'and', 'or', 'for', 'to', 'in', 'on', 'with', 'by'}
WORD_RE = re.compile(r"[a-zA-Z0-9]+(?:['\-][a-zA-Z0-9]+)*[?!]?")

def get_query_words(text, n=8):
    text = re.sub(r'[{}]', '', str(text or ''))
    all_words = WORD_RE.findall(text)

    def is_significant(word):
        base = word.rstrip('?!')
        if base.lower() in STOP_WORDS:
            return False
        if len(base) >= 3:
            return True
        has_letter = any(c.isalpha() for c in base)
        has_digit = any(c.isdigit() for c in base)
        return has_letter and has_digit

    significant = [word for word in all_words if is_significant(word)]
    return [word.lower() for word in (significant[:n] if len(significant) >= 3 else all_words[:n])]

title = citation['parsed_title']
print('Original title:')
print(title)
print('\nTokenized words:')
print(WORD_RE.findall(title))
print('\nQuery words used by lexical search:')
print(get_query_words(title, 8))

## 4. Open the SQLite DBs

In [ ]:
def sqlite_connect(path):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    return conn

arxiv_conn = sqlite_connect(ARXIV_DB)
dblp_conn = sqlite_connect(DBLP_DB)
openalex_conn = sqlite_connect(OPENALEX_DB)

print('Connections opened.')

## 5. Inspect the SQL idea

The lexical agent searches a helper table called `title_words`, then joins back to `records`.

It is roughly:

1. take query words
2. find records whose title contains those words
3. count how many matched
4. restrict by year if we have one
5. keep the strongest rows

In [ ]:
query_words = get_query_words(citation['parsed_title'], 8)
parsed_year = citation['parsed_year']
min_word_matches = 2 if len(query_words) >= 3 else 1
placeholders = ','.join('?' for _ in query_words)

sql_template = f'''
SELECT records.*, COUNT(*) AS matched_word_count
FROM title_words
JOIN records ON records.id = title_words.record_id
WHERE lower(title_words.word) IN ({placeholders})
AND (
    year = '' OR year IS NULL OR CAST(year AS INTEGER) BETWEEN ? AND ?
)
GROUP BY records.id
HAVING COUNT(*) >= ?
ORDER BY matched_word_count DESC
LIMIT ?
'''

print(sql_template)
print('\nParameters:')
print(query_words + [parsed_year - 1, parsed_year + 1, min_word_matches, 4])

## 6. Run raw SQL against arXiv

This is before lexical scoring. We are only seeing rows returned by the DB query.

In [ ]:
arxiv_sql = f'''
SELECT records.*, COUNT(*) AS matched_word_count
FROM title_words
JOIN records ON records.id = title_words.record_id
WHERE lower(title_words.word) IN ({placeholders})
AND (
    year = '' OR year IS NULL OR CAST(year AS INTEGER) BETWEEN ? AND ?
)
GROUP BY records.id
HAVING COUNT(*) >= ?
ORDER BY matched_word_count DESC
LIMIT ?
'''
params = query_words + [parsed_year - 1, parsed_year + 1, min_word_matches, 4]
arxiv_rows = arxiv_conn.execute(arxiv_sql, params).fetchall()
print('arXiv raw rows:', len(arxiv_rows))
for row in arxiv_rows[:3]:
    print(row['title'], '| matched_word_count =', row['matched_word_count'])

## 7. Run raw SQL against DBLP and OpenAlex

In [ ]:
dblp_sql = arxiv_sql.replace('year', 'year')
dblp_rows = dblp_conn.execute(dblp_sql, params).fetchall()
print('DBLP raw rows:', len(dblp_rows))
for row in dblp_rows[:3]:
    print(row['title'], '| matched_word_count =', row['matched_word_count'])

openalex_sql = arxiv_sql.replace('year', 'publication_year')
openalex_rows = openalex_conn.execute(openalex_sql, params).fetchall()
print('\nOpenAlex raw rows:', len(openalex_rows))
for row in openalex_rows[:3]:
    print(row['title'], '| matched_word_count =', row['matched_word_count'])

## 8. Rebuild the scoring logic in the notebook

Now we take a raw row and compute the same scoring components the lexical agent uses:

- `title_score`
- `author_score`
- `year_score`
- `lexical_score`

This is the real reranking step.

In [ ]:
SURNAME_PREFIXES = {'van', 'von', 'de', 'del', 'della', 'di', 'da', 'al', 'el', 'la', 'le', 'ben', 'ibn', 'mac', 'mc', 'o'}
NAME_SUFFIXES = {'jr', 'sr', 'ii', 'iii', 'iv', 'v'}

def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def normalize_title(text):
    return re.sub(r'[^a-z0-9]+', '', str(text or '').lower())

def split_authors(text):
    return [normalize_space(x) for x in str(text or '').split(';') if normalize_space(x)]

def parse_authors_loose(text):
    normalized = normalize_space(text)
    if not normalized:
        return []
    if ';' in normalized:
        return split_authors(normalized)
    if ' and ' in normalized.lower():
        normalized = re.sub(r'\s+(and|&)\s+', ';', normalized, flags=re.IGNORECASE)
        return [normalize_space(x) for x in normalized.split(';') if normalize_space(x)]
    return [normalize_space(x) for x in normalized.split(',') if normalize_space(x)]

def get_surname_from_parts(parts):
    while len(parts) >= 2 and parts[-1].lower().rstrip('.') in NAME_SUFFIXES:
        parts = parts[:-1]
    if len(parts) >= 3 and parts[-3].lower().rstrip('.') in SURNAME_PREFIXES:
        return ' '.join(parts[-3:])
    if len(parts) >= 2 and parts[-2].lower().rstrip('.') in SURNAME_PREFIXES:
        return ' '.join(parts[-2:])
    return parts[-1]

def normalize_author(name):
    name = normalize_space(name)
    if not name:
        return ''
    if ',' in name:
        parts = name.split(',', 1)
        surname = parts[0].strip().lower()
        initials = parts[1].strip() if len(parts) > 1 else ''
        first_initial = initials[0].lower() if initials else ''
        return f'{first_initial} {surname}'.strip()
    parts = [p for p in name.split() if p]
    surname = get_surname_from_parts(parts).lower()
    return f"{parts[0][0].lower()} {surname}".strip()

def author_overlap_score(ref_authors, cand_authors):
    ref_set = {normalize_author(a) for a in ref_authors if normalize_author(a)}
    cand_set = {normalize_author(a) for a in cand_authors if normalize_author(a)}
    if not ref_set or not cand_set:
        return 0.0
    return len(ref_set & cand_set) / max(1, len(ref_set))

def title_overlap_score(query_words, candidate_title):
    query_set = {w.lower() for w in query_words if w}
    candidate_set = set(get_query_words(candidate_title, 12))
    if not query_set or not candidate_set:
        return 0.0
    return len(query_set & candidate_set) / max(1, len(query_set))

def year_support_score(parsed_year, candidate_year):
    if parsed_year is None or candidate_year is None:
        return 0.5
    parsed_year = int(parsed_year)
    candidate_year = int(candidate_year)
    if parsed_year == candidate_year:
        return 1.0
    if abs(parsed_year - candidate_year) <= 1:
        return 0.7
    if abs(parsed_year - candidate_year) <= 2:
        return 0.3
    return 0.0

## 9. Score one arXiv row manually

Take the first arXiv result and compute the ranking signals ourselves.

In [ ]:
row = arxiv_rows[0]
candidate_title = row['title']
candidate_authors = parse_authors_loose(row['authors'])
candidate_year = int(row['year']) if str(row['year']).isdigit() else None

title_score = title_overlap_score(query_words, candidate_title)
author_score = author_overlap_score(citation['parsed_authors'], candidate_authors)
year_score = year_support_score(citation['parsed_year'], candidate_year)
normalized_title_exact = normalize_title(citation['parsed_title']) == normalize_title(candidate_title)

if normalized_title_exact:
    title_score = 1.0

lexical_score = 0.70 * title_score + 0.20 * author_score + 0.10 * year_score
if normalized_title_exact:
    lexical_score += 0.10
lexical_score = min(1.0, lexical_score)

pprint({
    'candidate_title': candidate_title,
    'candidate_authors': candidate_authors,
    'candidate_year': candidate_year,
    'normalized_title_exact': normalized_title_exact,
    'title_score': round(title_score, 4),
    'author_score': round(author_score, 4),
    'year_score': round(year_score, 4),
    'lexical_score': round(lexical_score, 4),
})

## 10. Score the top rows from each DB

Now do the same for arXiv, DBLP, and OpenAlex so we can compare them directly.

In [ ]:
def row_to_candidate(row, db_name):
    if db_name == 'arxiv':
        authors = parse_authors_loose(row['authors'])
        year = int(row['year']) if str(row['year']).isdigit() else None
        record_id = row['arxiv_id']
        venue = row['venue']
    elif db_name == 'dblp':
        authors = split_authors(row['authors'])
        year = int(row['year']) if str(row['year']).isdigit() else None
        record_id = row['dblp_key']
        venue = row['venue']
    else:
        authors = split_authors(row['authors'])
        year = int(row['publication_year']) if str(row['publication_year']).isdigit() else None
        record_id = row['openalex_id']
        venue = row['venue']

    title = row['title']
    title_score = title_overlap_score(query_words, title)
    author_score = author_overlap_score(citation['parsed_authors'], authors)
    year_score = year_support_score(citation['parsed_year'], year)
    normalized_title_exact = normalize_title(citation['parsed_title']) == normalize_title(title)
    if normalized_title_exact:
        title_score = 1.0
    lexical_score = 0.70 * title_score + 0.20 * author_score + 0.10 * year_score
    if normalized_title_exact:
        lexical_score += 0.10
    lexical_score = min(1.0, lexical_score)

    return {
        'db': db_name,
        'record_id': record_id,
        'title': title,
        'authors': authors,
        'year': year,
        'venue': venue,
        'title_score': round(title_score, 4),
        'author_score': round(author_score, 4),
        'year_score': round(year_score, 4),
        'lexical_score': round(lexical_score, 4),
        'normalized_title_exact': normalized_title_exact,
    }

manual_candidates = []
for db_name, rows in [('arxiv', arxiv_rows), ('dblp', dblp_rows), ('openalex', openalex_rows)]:
    for row in rows[:2]:
        cand = row_to_candidate(row, db_name)
        manual_candidates.append(cand)

manual_candidates = sorted(
    manual_candidates,
    key=lambda c: (c['normalized_title_exact'], c['lexical_score']),
    reverse=True,
)
pprint(manual_candidates[:6])

## 11. How lexical decides what is strong or weak

The current lexical policy is:

- if DB1 top candidate is near-perfect, stop early
- otherwise query the next DB
- keep at most 2 candidates in the final list

Current thresholds:

- near-perfect:
  - `title_score >= 0.85`
  - `author_score >= 0.50`
  - `lexical_score >= 0.75`
- weak:
  - `title_score < 0.75`
  - or `lexical_score < 0.65`

In [ ]:
top = manual_candidates[0]
near_perfect = (
    top['title_score'] >= 0.85 and
    top['author_score'] >= 0.50 and
    top['lexical_score'] >= 0.75
)
weak = (
    top['title_score'] < 0.75 or
    top['lexical_score'] < 0.65
)

print('Top candidate title:', top['title'])
print('Near-perfect?', near_perfect)
print('Weak?', weak)

## 12. What fusion would see

Fusion does not do lexical search itself. It receives the ranked lexical list and usually selects candidate 1.

- candidate 1 = selected candidate
- candidate 2 = backup only if scores are very close

In [ ]:
top2 = manual_candidates[:2]
for i, cand in enumerate(top2, start=1):
    print(f'Rank {i}: {cand["db"]} | {cand["title"]} | lexical_score={cand["lexical_score"]}')

if len(top2) == 2:
    score_gap = round(top2[0]['lexical_score'] - top2[1]['lexical_score'], 4)
    print('\nScore gap:', score_gap)
    print('Keep backup?', score_gap <= 0.05)

## 13. Cleanup

In [ ]:
arxiv_conn.close()
dblp_conn.close()
openalex_conn.close()
print('Connections closed.')